# Geodesic Path Demo

Compute the exact geodesic (on-surface) path between two regions with `subject.calculate_geodesic_path`, write it as a label, and plot it **interactively** with the package's `ScalpelVisualizer` (trimesh scene — rotate/zoom inline).

In [ ]:
import os
import numpy as np
from scalpel.subject import ScalpelSubject
from scalpel.utils import surface_utils

# FreeSurfer subjects dir: use $FREESURFER_HOME if set, else the default install
fs_home = os.environ.get("FREESURFER_HOME", "/Applications/freesurfer/8.2.0")
subjects_dir = os.path.join(fs_home, "subjects")
subject = ScalpelSubject(subject_id="fsaverage", hemi="lh",
                         subjects_dir=subjects_dir, surface_type="inflated")
V = subject.surface_RAS
print(f"loaded {subject.subject_id} {subject.hemi}: {len(V):,} vertices")


### Create two endpoint labels (frontal & occipital poles)

In [ ]:
frontal = np.where(V[:, 1] > V[:, 1].max() - 15)[0]
occipital = np.where(V[:, 1] < V[:, 1].min() + 15)[0]
subject.load_label("frontal", label_idxs=frontal, label_RAS=V[frontal])
subject.load_label("occipital", label_idxs=occipital, label_RAS=V[occipital])


### Compute the geodesic path and write it as a label

In [ ]:
path_vertices, length = subject.calculate_geodesic_path("frontal", "occipital")
print(f"geodesic path: {len(path_vertices)} vertices, {length:.1f} mm")

# widen the 1-vertex path into a visible ribbon so it renders as a surface patch
ribbon = path_vertices
for _ in range(2):
    ribbon = surface_utils.find_adjacent_indices(ribbon, subject.faces).astype(int)
subject.load_label("geodesic_path", label_idxs=ribbon, label_RAS=V[ribbon])

# (optional) save it as a FreeSurfer .label file
# subject.labels["geodesic_path"].write_label("geodesic_path", overwrite=True)


### Interactive plot\n\nRotate / zoom the scene inline. The path (red) follows the folded surface.

In [ ]:
subject.plot(view="lateral", labels=["geodesic_path"])
